In [799]:
import numpy as np
import pandas as pd
import plotly.express as px
import collections
import itertools

In [800]:
np.random.seed(0)

In [801]:
def bayesian_update(priors):
    if np.sum(priors) == 0:
        return np.zeros_like(priors)
    return priors / np.sum(priors)

In [802]:
def best_response(x, thresholds, priors, c):
    posteriors = bayesian_update(priors)
    search_space = [x]
    search_space.extend(thresholds[thresholds > x].tolist())
    utilities = []
    for x_p in search_space:
        utility = np.dot(posteriors, x_p >= thresholds)
        cost = c * abs(x-x_p)
        utilities.append(utility - cost)
    return search_space[np.argmax(utilities)]

In [803]:
def manipulation_thresholds(thresholds, priors, c):
    idx = np.argsort(thresholds)
    thresholds = thresholds[idx]
    priors = priors[idx]
    if priors.sum() == 0:
        return thresholds, priors, thresholds

    manip_thresholds = np.maximum(0, thresholds - (bayesian_update(priors) / c))

    if len(manip_thresholds) <= 1:
        return manip_thresholds, priors, thresholds

    merged_priors = [priors[0]]
    merged_thresholds = [thresholds[0]]
    merged_manip_thresholds = [manip_thresholds[0]]

    for i in range(1, len(manip_thresholds)):
        if manip_thresholds[i] <= merged_manip_thresholds[-1]:
            merged_priors[-1] += priors[i]
            merged_thresholds[-1] = thresholds[i]
            merged_manip_thresholds[-1] = manip_thresholds[i]
        else:
            merged_priors.append(priors[i])
            merged_thresholds.append(thresholds[i])
            merged_manip_thresholds.append(manip_thresholds[i])

    return (
        np.array(merged_manip_thresholds),
        np.array(merged_priors),
        np.array(merged_thresholds),
    )

In [804]:
def balance_priors(priors, random=True):
    total = np.sum(priors)
    if total == 1:
        return priors
    indices = priors == 0
    remainder = 1 - total
    if random:
        p = np.random.rand(indices.sum())
        p = (p / p.sum()) * remainder
    else:
        p = remainder / indices.sum()
    priors[indices] = p
    return priors

In [805]:
def accuracy_loss_cont(thresholds, manip_thresholds, threshold_true):
    losses = []
    for i in range(len(thresholds)):
        if thresholds[i] < threshold_true:
            loss = (threshold_true - manip_thresholds[i])
        else:
            loss = np.abs(manip_thresholds[i] - threshold_true)
        losses.append(loss)
    return np.array(losses)

In [806]:
def accuracy_loss_disc(X, X_p, thresholds, threshold_true):
    Y_true = (X >= threshold_true).astype(float)
    losses = []
    for threshold in thresholds:
        Y_p = (X_p >= threshold).astype(float)
        acc_loss = np.abs(Y_true - Y_p).mean()
        losses.append(acc_loss)
    return np.array(losses)

In [807]:
def evaluate_partition(partition, thresholds, priors, threshold_true, c):
    threshold_p = thresholds[partition]
    priors_p = priors[partition]
    manip_threshold_p, priors_p, threshold_p = manipulation_thresholds(threshold_p, priors_p, c)
    acc_loss = accuracy_loss_cont(threshold_p, manip_threshold_p, threshold_true)
    return priors_p, acc_loss

In [808]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return

    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

In [809]:
def find_partitions_greedy(thresholds, priors, threshold_true, c):
    partitions = [[i] for i in range(len(priors))]
    P = {}
    next_id = 0

    P_init = [list(block) for block in partitions]
    for block in P_init:
        P[next_id] = list(block)
        next_id += 1

    active_ids = set(P.keys())

    Q = collections.deque(itertools.combinations(active_ids, 2))

    while Q:
        a_id, b_id = Q.popleft()

        # Skip if one was merged already
        if a_id not in active_ids or b_id not in active_ids:
            continue

        a = P[a_id]
        b = P[b_id]

        priors_a, acc_loss_a = evaluate_partition(a, thresholds, priors, threshold_true, c)
        priors_b, acc_loss_b = evaluate_partition(b, thresholds, priors, threshold_true, c)

        lhs = np.dot(priors_a, acc_loss_a) + np.dot(priors_b, acc_loss_b)
        ab = sorted(a + b)
        priors_ab, acc_loss_ab = evaluate_partition(ab, thresholds, priors, threshold_true, c)
        rhs = np.dot(priors_ab, acc_loss_ab)

        if lhs > rhs:
            # -------- MERGE --------
            # Remove a and b from partition
            active_ids.remove(a_id)
            active_ids.remove(b_id)
            del P[a_id]
            del P[b_id]

            # Remove all (a, c) and (b, d) from Q
            Q = collections.deque(
                (x, y)
                for (x, y) in Q
                if x not in {a_id, b_id} and y not in {a_id, b_id}
            )

            # Add merged block
            new_id = next_id
            next_id += 1
            P[new_id] = ab
            active_ids.add(new_id)

            # Insert (ab, c) for all remaining c in P \ {a,b}
            for c_id in active_ids:
                if c_id != new_id:
                    Q.append((new_id, c_id))
    return list(P.values())

In [810]:
def find_partitions_optimal(thresholds, priors, threshold_true, c):
    indices = [i for i in range(len(thresholds))]

    parts = set_partitions(indices)
    partitions_set = []
    for part in parts:
        partitions_set.append(part)


    best_partition = None
    best_loss = np.inf

    for partitions in partitions_set:
        acc_loss = 0.
        for partition in partitions:
            priors_p, acc_loss_p = evaluate_partition(partition, thresholds, priors, threshold_true, c)
            acc_loss += np.dot(priors_p, acc_loss_p)

        if acc_loss < best_loss:
            best_loss = acc_loss
            best_partition = partitions

    return best_partition

In [811]:
c = 5.
threshold_true = 0.5

threshold_min, threshold_max, threshold_delta = 0., 1., 0.1
thresholds = np.arange(threshold_min+threshold_delta, threshold_max, threshold_delta).round(4)

priors = np.zeros_like(thresholds)
priors[6] = 0.28
priors[7] = 0.31
priors[8] = 0.21

# priors = np.array([0.11393634, 0.11459784, 0.03594993, 0.25, 0.02203078, 0.05390004, 0.06215049, 0.09743458, 0.25])
balance_priors(priors, random=True)

print(np.sum(priors))
pd.DataFrame({"threshold": thresholds, "priors": priors}).round(3).T

1.0


,0,1,2,3,4,5,6,7,8
threshold,0.100,0.200,0.300,0.400,0.500,0.600,0.70,0.80,0.90
priors,0.032,0.041,0.035,0.031,0.024,0.037,0.28,0.31,0.21


In [812]:
partition_greedy = find_partitions_greedy(thresholds, priors, threshold_true, c)
partition_optimal = find_partitions_optimal(thresholds, priors, threshold_true, c)

In [813]:
acc_loss_greedy = 0.
for partition in partition_greedy:
    priors_p, acc_loss_p = evaluate_partition(partition, thresholds, priors, threshold_true, c)
    acc_loss_greedy += np.dot(priors_p, acc_loss_p)

In [814]:
acc_loss_optimal = 0.
for partition in partition_optimal:
    priors_p, acc_loss_p = evaluate_partition(partition, thresholds, priors, threshold_true, c)
    acc_loss_optimal += np.dot(priors_p, acc_loss_p)

In [815]:
print("Greedy")
print("------")
print(f"Partition: {partition_greedy}")
print(f"Acc Loss : {acc_loss_greedy:.4f}")
print()
print("Optimal")
print("-------")
print(f"Partition: {partition_optimal}")
print(f"Acc Loss : {acc_loss_optimal:.4f}")

Greedy
------
Partition: [[6], [7], [8], [0, 1, 2, 3, 4, 5]]
Acc Loss : 0.1158

Optimal
-------
Partition: [[0, 1, 2, 3, 4, 5], [6], [7], [8]]
Acc Loss : 0.1158


In [816]:
# a = [2,3]
# b = [0,1]
b = [2,3]
a = [0,1]
pa, la = evaluate_partition(a, thresholds, priors, threshold_true, c)
pb, lb = evaluate_partition(b, thresholds, priors, threshold_true, c)
pab, lab = evaluate_partition(a+b, thresholds, priors, threshold_true, c)

np.dot(pa,la)+np.dot(pb, lb), np.dot(pab, lab)

(np.float64(0.048992899451930985), np.float64(0.04201272176438632))

In [817]:
priors[a+b], priors[b+a]

(array([0.03153015, 0.04108869, 0.03462965, 0.03130435]),
 array([0.03462965, 0.03130435, 0.03153015, 0.04108869]))